In [ ]:
!git clone https://github.com/piotrszczypior/backdoor-resnet.git
!cd backdoor-resnet & git checkout tsne

In [ ]:
import sys
import os

notebook_dir = os.path.abspath(".")
project_path = os.path.join(notebook_dir, "backdoor-resnet")
sys.path.append(project_path)

In [ ]:
!mkdir -p weights
!gdown https://drive.google.com/drive/u/1/folders/1dKsJ8GthFI31lvWMcqxmOxlxG6CBasrC --folder --output weights

In [ ]:
import cv2
import numpy as np

import src.loader
from src.backdoor import white_box_trigger
from src.utils import compute_gradcam

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def get_model():
    model = get_resnet_model(10)
    checkpoint = torch.load(
        "weights/weights-square-trigger.pth",
        map_location=DEVICE,
    )
    model.to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    model.to(DEVICE)

    return model


dataset = BackdooredDataset(
        dataset="CIFAR10",
        train=False,
        transform=loader.get_test_transform_cifar10(),
        backdoor=True,
        trigger_fn=white_box_trigger,
        mode="replace",
        label_mode="clean_label",
        p=1,
)

idx = 32
tensor_input = dataset[idx]
img = dataset.get_img_pil(idx)

gradcam = compute_gradcam(model, tensor_input, np.array(img))

cv2.imwrite('cifar10_square_gradcam.png', gradcam)